# 10 · Embeddings: Choosing and Evaluating a Model

**Time:** about 60 minutes · **Prerequisites:** `09_RAG_from_Scratch`

> **This notebook steps out of MLX.** It uses
> [`sentence-transformers`](https://www.sbert.net/) (a PyTorch library) for its
> pretrained sentence encoders. Device selection is automatic: inspect
> `model.device` to see whether a run uses MPS (Apple's GPU) or the CPU.
> Trade-offs: high-quality pretrained models and easy APIs, but a larger
> install (`torch`) and weights that do not load into MLX directly.

## Learning objectives
1. Say what an embedding is, and **look at an embedding space**.
2. Rerun notebook 09's **retrieval evaluation** with a learned embedding model
   and see which failures it fixes.
3. **Compare models** on quality and speed using the same labeled queries.
4. Use embeddings as **features** for classification, and compare with the
   LSTM from notebook 01.
5. Know the limits: scores are not calibrated, and the evaluation set here is small.

First run downloads `all-MiniLM-L6-v2` (about 90 MB). The optional larger
model (`all-mpnet-base-v2`, about 420 MB) is switched off by default.

In [ ]:
import json
import re
import time
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE

from mlx_nlp_utils import (
    bootstrap_ci, group_train_val_split, load_rag_eval_queries, load_rag_knowledge_base,
    sanity_check, tokenize,
)

sns.set_style('whitegrid')
warnings.filterwarnings('ignore', category=FutureWarning, module='sklearn')   # sklearn announces API changes for LogisticRegressionCV

## Part 1: What is an embedding?

An embedding maps text to a point in a vector space so that texts with
similar meaning land near each other. Unlike word counts (one dimension per
vocabulary word), the dimensions are **learned** by training on large amounts of text:

```
"The cat sleeps" → [0.23, -0.15, 0.67, …, 0.42]     (384 numbers)
```

Similarity is the cosine of the angle between vectors; for unit-length
vectors it is a dot product, exactly as in notebook 09.

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embedding_dim = getattr(model, "get_embedding_dimension", model.get_sentence_embedding_dimension)()
print(f"{embedding_dim} dimensions, running on {model.device}")

sentences = {
    "Animals": ["The cat sits on the mat", "Dogs are loyal companions", "The bird flew over the tree",
                "Lions hunt in the savannah"],
    "Technology": ["The computer crashed again", "Python is a great programming language",
                   "Artificial intelligence is the future", "My smartphone battery is dead"],
    "Food": ["The pizza tastes delicious", "I love eating sushi for dinner",
             "Vegetables are healthy for you", "The cake was too sweet"],
    "Emotions": ["I feel so happy today", "This makes me very angry", "She is sad about the news",
                 "They feel overwhelmed"],
}
texts = [s for group in sentences.values() for s in group]
groups = [g for g, items in sentences.items() for _ in items]
vectors = model.encode(texts, normalize_embeddings=True)
print("embedding matrix:", vectors.shape)

### Look at it, and don't over-read the picture

A 384-dimensional space can't be drawn, so we project it to 2D. Two projections:
**PCA** (a linear projection onto the directions of greatest variance) and
**t-SNE** (which preserves local neighborhoods but distorts global distances
and depends on its random seed and "perplexity" setting).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
palette = dict(zip(sentences, ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']))
for ax, (name, points) in zip(axes, [
        ("PCA", PCA(n_components=2).fit_transform(vectors)),
        ("t-SNE (perplexity 5)", TSNE(n_components=2, perplexity=5, random_state=0).fit_transform(vectors))]):
    for (x, y), text, group in zip(points, texts, groups):
        ax.scatter(x, y, c=palette[group], s=120, edgecolors='black', alpha=0.7)
        ax.annotate(text[:24], (x, y), xytext=(4, 4), textcoords='offset points', fontsize=8)
    ax.set_title(name)
plt.tight_layout(); plt.show()

The clusters are visible, but remember that this is 16 hand-picked
sentences from four obviously different topics, and a 2D picture of 384
dimensions loses information. A picture is not an evaluation.
Here is a number instead: **how often is a sentence's nearest neighbor in
the same topic?**

In [ ]:
sim = vectors @ vectors.T
np.fill_diagonal(sim, -1)
nearest = sim.argmax(axis=1)
same_topic = np.mean([groups[i] == groups[j] for i, j in enumerate(nearest)])
print(f"Nearest neighbor shares the topic for {same_topic:.0%} of the 16 sentences")

## Part 2: Rerun the retrieval evaluation

Notebook 09 scored word-overlap retrieval on 40 labeled questions over a
50-document knowledge base and found that **paraphrases fail**. Now the same
evaluation, with learned embeddings.

Two practical details:

* Sentence-transformers models expose `encode_query` and `encode_document`.
  Some models use different prompts or prefixes for the two sides;
  these methods apply them when the model defines them.
* We normalize embeddings so the dot product is cosine similarity.

In [ ]:
knowledge_base = load_rag_knowledge_base()
eval_queries = load_rag_eval_queries()
queries = [q["query"] for q in eval_queries]
kinds = np.array([q["kind"] for q in eval_queries])
relevant = np.array([q["relevant_doc"] for q in eval_queries])

def ranks_from_scores(scores):
    """Rank (1 = best) of the correct document for each query."""
    return np.array([1 + int((row > row[rel]).sum()) for row, rel in zip(scores, relevant)])

def summarize(name, ranks):
    print(name)
    out = {}
    for kind in ("lexical", "paraphrase"):
        r = ranks[kinds == kind]
        out[kind] = dict(r1=np.mean(r <= 1), r3=np.mean(r <= 3), mrr=np.mean(1 / r))
        print(f"   {kind:<11} recall@1 {out[kind]['r1']:.0%}   recall@3 {out[kind]['r3']:.0%}   MRR {out[kind]['mrr']:.2f}")
    return out

# Notebook 09's TF-IDF retriever, in scikit-learn form, as the baseline to beat.
tfidf = TfidfVectorizer(token_pattern=r"[a-z0-9]+").fit(knowledge_base)
tfidf_scores = (tfidf.transform(queries) @ tfidf.transform(knowledge_base).T).toarray()
tfidf_results = summarize("TF-IDF (notebook 09)", ranks_from_scores(tfidf_scores))

def embedding_scores(model):
    docs = model.encode_document(knowledge_base, normalize_embeddings=True)
    qs = model.encode_query(queries, normalize_embeddings=True)
    return qs @ docs.T

minilm_scores = embedding_scores(model)
minilm_ranks = ranks_from_scores(minilm_scores)
minilm_results = summarize("MiniLM (384-d)", minilm_ranks)
sanity_check(minilm_results["paraphrase"]["r3"] > tfidf_results["paraphrase"]["r3"],
             "learned embeddings retrieve more paraphrase queries in the top 3 than TF-IDF")

Paraphrase retrieval improves clearly: recall@1 rises from about 35% to
55%, and recall@3 from about 55% to 80%. The model has seen "RAM" and "memory"
in similar contexts, so it can connect questions to documents without shared
words. Lexical queries were already near ceiling. It is an improvement, **not
a solution**: nearly half the paraphrases still miss at rank 1.

### What still fails?

In [ ]:
for q, rank, row in zip(eval_queries, minilm_ranks, minilm_scores):
    if rank > 1:
        top = int(row.argmax())
        print(f"[{q['kind']}] {q['query']!r}\n   correct doc ranked {rank}; top hit: {knowledge_base[top][:80]}...\n")

Most misses rank the right document 2nd to 5th, behind a *neighboring topic*:
a question about "how to word instructions" retrieves the few-shot or
chain-of-thought documents, which really are about related ideas. A few miss
badly (the "facts newer than its training cutoff" question never finds the RAG
document). Some of these are ambiguity in the questions as much as model
failure. Note the **uncertainty**: 20 queries per kind means each query is 5
points, so differences of one or two queries are noise.

## Part 3: Can you refuse out-of-scope questions?

Notebook 09 found that word-overlap scores could not separate covered from
uncovered questions. Do learned embeddings do better?

In [ ]:
out_of_scope = ["How do I cook pasta?", "Who won the 1998 World Cup?", "What is the boiling point of water?",
                "Recommend a hiking trail near Denver", "How do I change a car tire?",
                "What is the population of Canada?", "Best way to learn the guitar?", "How tall is Mount Everest?"]
docs = model.encode_document(knowledge_base, normalize_embeddings=True)
in_top = minilm_scores.max(axis=1)
out_top = (model.encode_query(out_of_scope, normalize_embeddings=True) @ docs.T).max(axis=1)

plt.figure(figsize=(8, 3.5))
bins = np.linspace(0, 0.8, 25)
plt.hist(in_top, bins=bins, alpha=0.7, label='covered questions')
plt.hist(out_top, bins=bins, alpha=0.7, label='uncovered questions')
plt.xlabel('top cosine similarity (MiniLM)'); plt.ylabel('queries'); plt.legend()
plt.title('Do covered and uncovered questions separate?'); plt.show()

best = max(((np.mean(in_top >= th) - np.mean(out_top >= th), th) for th in np.linspace(0.05, 0.6, 56)))
print(f"Best single threshold ≈ {best[1]:.2f}: answers {np.mean(in_top >= best[1]):.0%} of covered "
      f"and {np.mean(out_top >= best[1]):.0%} of uncovered questions")

Much better separation than word counts: at the best single threshold (about
0.2) the system answers nearly all covered questions and refuses nearly all
of these eight uncovered ones. But eight uncovered questions is a small test,
and the threshold is specific to this model and corpus: cosine similarity values from
different models are not comparable, and they are **not probabilities**. Choose
the threshold on labeled data for the model you deploy, and re-check it
whenever you change the model or the documents.

## Part 4: Compare models: quality and speed

A bigger model is not automatically better *for your data*. To compare,
use the same labeled queries and measure time. We add an optional second
model; set `COMPARE_MPNET = True` to download and test it.

In [ ]:
COMPARE_MPNET = False
candidates = {"MiniLM (384-d)": model}
if COMPARE_MPNET:
    candidates["MPNet (768-d)"] = SentenceTransformer('all-mpnet-base-v2')

def time_encoding(m, corpus, repeats=3):
    m.encode(corpus[:8])                                        # warm-up
    times = []
    for _ in range(repeats):
        start = time.perf_counter(); m.encode(corpus, batch_size=32); times.append(time.perf_counter() - start)
    return len(corpus) / min(times)

corpus = knowledge_base * 4                                    # 200 short texts
print(f"{'model':<18} {'dim':>4} {'paraphrase r@1':>15} {'paraphrase MRR':>15} {'texts/sec':>10}")
for name, m in candidates.items():
    r = ranks_from_scores(embedding_scores(m))
    para = r[kinds == 'paraphrase']
    dim = getattr(m, "get_embedding_dimension", m.get_sentence_embedding_dimension)()
    print(f"{name:<18} {dim:>4} {np.mean(para <= 1):>15.0%} "
          f"{np.mean(1 / para):>15.2f} {time_encoding(m, corpus):>10.0f}")

Read the table with the sample size in mind. With 20 paraphrase queries,
two models that differ by one or two queries are tied. A larger model costs
more to run and uses more memory; it needs to earn that with a **clear** gain
on *your* queries. Everything on this page, including the "right" model, is
specific to the evaluation set.

## Part 5: Embeddings as features for classification

Pretrained embeddings also make useful **features**: encode each text once,
then fit a simple classifier on the vectors. Notebook 01's FAQ promised this
comparison. We use the same grouped splits as notebook 01 and compare:

* **TF-IDF + logistic regression** (word counts),
* **MiniLM + logistic regression with sklearn's default `C=1`**,
* **MiniLM + logistic regression with `C` chosen by cross-validation** on the
  training set only. The folds are *grouped* by base phrase, for the same
  leakage reason as notebook 01; otherwise near-copies in different folds would
  favor settings that memorize.

`C` is the inverse of regularization strength: small `C` means a strong pull
toward small weights. Unit-length embeddings have small numbers in each
coordinate, so they often need a larger `C` than the default.

🤔 **Predict:** will the default `C=1` do well on all three splits?

In [ ]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import GroupKFold

def base_phrase(text):
    phrase = " ".join(tokenize(text))
    for word in ["please", "could you", "can you", "hey", "ok"]:
        if phrase.startswith(word + " "):
            phrase = phrase[len(word) + 1:]
        if phrase.endswith(" " + word):
            phrase = phrase[: -len(word) - 1]
    return phrase

data = json.loads(Path('../data/intent_samples/data.json').read_text())
pairs = list(dict.fromkeys(zip(data['texts'], data['labels'])))
rows = []
for seed in range(3):
    train, val = group_train_val_split(pairs, [base_phrase(t) for t, _ in pairs], 0.2, seed)
    tr_t, tr_l = map(list, zip(*train)); va_t, va_l = map(list, zip(*val))
    X_tr, X_va = (model.encode(x, normalize_embeddings=True) for x in (tr_t, va_t))

    default = LogisticRegression(max_iter=3000).fit(X_tr, tr_l).score(X_va, va_l)
    folds = GroupKFold(n_splits=3).split(X_tr, tr_l, [base_phrase(t) for t in tr_t])
    tuned_clf = LogisticRegressionCV(Cs=[1, 10, 100, 1000], cv=list(folds), max_iter=3000).fit(X_tr, tr_l)
    tuned = tuned_clf.score(X_va, va_l)

    vec = TfidfVectorizer(tokenizer=tokenize, token_pattern=None, ngram_range=(1, 2)).fit(tr_t)
    tfidf_acc = LogisticRegression(max_iter=1000).fit(vec.transform(tr_t), tr_l).score(vec.transform(va_t), va_l)
    rows.append((seed, len(va_t), Counter(tr_l).most_common(1)[0][0], Counter(va_l).most_common(1)[0][0],
                 tfidf_acc, default, tuned, float(tuned_clf.C_[0])))

print(f"{'seed':>4} {'n_val':>5} {'TF-IDF':>7} {'MiniLM C=1':>11} {'MiniLM tuned':>13} {'chosen C':>9}   most common label: train / val")
for seed, n, tr_major, va_major, tf_acc, default, tuned, C in rows:
    print(f"{seed:>4} {n:>5} {tf_acc:>7.0%} {default:>11.0%} {tuned:>13.0%} {C:>9g}   {tr_major} / {va_major}")
print("\nFor context (different training budgets, so only roughly comparable): notebook 01's LSTM scored about "
      "80-82% on these splits; notebook 05's longer-trained LSTM and Transformer scored 92% and 97% on average.")

Two lessons here, one about embeddings and one about evaluation:

1. **Defaults matter.** With `C=1` the classifier is over-regularized on
   unit-length embeddings and, on a split where the label mix changes
   (look at the "most common label" columns), it leans toward the majority
   training class and does poorly on one split. Choosing `C` by grouped
   cross-validation on the *training* data fixes that without touching the
   validation set. The cost: a second hyperparameter to tune carefully, and the
   same noisy 20-example validation sets as before.
2. **Pretrained features are competitive, not magic.** After tuning, MiniLM + a
   linear classifier is in the same range as TF-IDF and as the neural models
   from notebooks 01 and 05; with this little and this easy data, no method is
   clearly ahead. Embeddings help most when test inputs contain words the
   training set never saw ("play", "window"), since the encoder has met them in
   its own pretraining. Their advantage over TF-IDF should grow with
   vocabulary diversity (try a real dataset like SNIPS via `make setup-real`).

## Part 6: Pitfalls

* **Query vs document.** Use `encode_query` / `encode_document` (or the
  model's documented prefixes). Mixing them up quietly hurts quality.
* **Normalization.** If you compare with a dot product, normalize first.
* **Truncation.** Models cap input length (256 tokens for MiniLM); longer
  text is silently cut. Chunk long documents.
* **Scores are not calibrated.** A cosine of 0.5 means different things for
  different models and corpora.
* **Domain shift.** A model trained on web text may not know your jargon.
  Evaluate on your queries, and consider fine-tuning only if the evaluation says so.
* **Index/model coupling.** Document vectors from one model can't be searched
  with queries from another; changing the model means re-embedding everything.

## Summary
* Learned embeddings fix the paraphrase failures of word-overlap retrieval.
* Evaluate with labeled queries (recall@k, MRR), split by query type, and
  keep the sample size in mind.
* Cosine scores need a data-chosen threshold and don't transfer across models.
* Pretrained embeddings + a linear classifier is a cheap baseline worth
  trying, but tune its regularization (on training data only) and compare
  against TF-IDF before believing it is better.

## 🧠 Exercises

**1. Write harder queries.** Add 10 paraphrase queries of your own to
`data/rag_samples/eval_queries.json` (or to `eval_queries` in memory). Which
fail, and what do they have in common?

<details><summary>What to look for</summary>

Queries that require reasoning, negation ("which method does *not* use
memory?"), or multiple documents tend to fail. Retrieval by similarity finds
topics, not logical answers; that is where re-ranking and LLM-based query
rewriting come in.
</details>

**2. Chunk size.** Concatenate every 5 documents into one longer chunk and
re-evaluate. 🤔 Predict: does recall@1 go up or down?

<details><summary>What to look for</summary>

Bigger chunks make it likelier that the right *text* is retrieved, but the
labeled "relevant document" now sits inside a chunk with four others, and the
embedding becomes a blur of five topics, so the match is weaker. Chunk size is
a retrieval hyperparameter to tune against an evaluation set.
</details>

**3. Hybrid scoring.** Combine scores: `0.5 * tfidf_scores + 0.5 * minilm_scores`
(after scaling each to its range). Does the combination beat both?

<details><summary>What to look for</summary>

Often slightly, particularly for queries with rare exact terms. That is the
idea behind hybrid search. With 40 queries the difference is small enough to
be noise, so you would want a bigger evaluation set to decide.
</details>

## ❓ FAQ

**Why not train embeddings in MLX?** You can (it is the same training loop
as notebook 05 with a contrastive loss), but pretrained encoders have seen
orders of magnitude more data than we could use here.

**How are these models trained?** Typically with a *contrastive* loss: pull
the embeddings of matching pairs (a question and its answer) together and
push non-matching pairs apart.